# Funções em C

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Declarar funções com protótipos, e explicar por que um argumento alterado dentro da função continua igual fora (passagem por valor).
- Guardar estado entre chamadas com uma variável static, e dividir um problema em funções pequenas.
- Escrever funções recursivas, inclusive funções que chamam umas às outras.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Uma função: tipo, nome, parâmetros
2. Declare antes de usar: protótipos
3. (void), e a armadilha dos parênteses vazios
4. Argumentos são cópias
5. Variáveis locais, globais e static
6. Funções que chamam funções, e recursão
7. A biblioteca padrão

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check_int e check_str imprimem ✓ ou ✗; os exercícios abaixo usam
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado %ld, veio %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado \"%s\", veio \"%s\"\n", what, expected, actual);
    }
}

## 1. Uma função: tipo, nome, parâmetros

Uma função em C parece um método `static` do Java sem classe em volta: um **tipo de retorno**, um nome, **parâmetros com tipo** entre parênteses e um corpo. O `return` devolve o resultado. Uma função que não devolve nada tem o tipo `void`.

In [ ]:
#include <stdio.h>

int square(int x) {
    return x * x;
}

void print_line(int width) {
    for (int i = 0; i < width; i++) {
        printf("-");
    }
    printf("\n");
}

int main(void) {
    print_line(12);
    printf("5 ao quadrado é %d\n", square(5));
    printf("square(square(3)) é %d\n", square(square(3)));
    print_line(12);
    return 0;
}

🧪 **Experimente:** escreva `int cube(int x)` usando `square`, e imprima `cube(4)`.

## 2. Declare antes de usar: protótipos

O compilador lê o arquivo **de cima para baixo**. Quando chega numa chamada, ele já precisa conhecer a função: o tipo de retorno e os parâmetros. Ou defina a função acima da chamada, ou ponha um **protótipo** no topo, a primeira linha da função seguida de `;`, e defina-a em qualquer lugar abaixo. Os protótipos deixam o `main` no topo, onde é fácil de achar.

In [ ]:
#include <stdio.h>

double average(int a, int b);      // protótipo: "esta função existe"

int main(void) {
    printf("%.1f\n", average(7, 8));
    return 0;
}

double average(int a, int b) {     // a definição, abaixo
    return (a + b) / 2.0;
}

🧪 **Experimente:** apague a linha do protótipo e leia o erro: o compilador encontrou `average` antes de conhecê-la.

## 3. (void), e a armadilha dos parênteses vazios

Para uma função sem parâmetros, escreva `(void)`: `int next_ticket(void)`. No C17, um `()` vazio numa declaração quer dizer "parâmetros não informados", então o compilador não consegue conferir se as chamadas passam os argumentos certos. Do mesmo jeito, uma função que devolve um valor precisa ter `return` em todo caminho: chegar ao fim sem ele entrega lixo para quem chamou (e gera um aviso).

In [ ]:
#include <stdio.h>

int lucky(void) {
    return 7;
}

const char *sign(int n) {
    if (n < 0) {
        return "negativo";
    }
    if (n > 0) {
        return "positivo";
    }
    return "zero";
}

int main(void) {
    printf("%d %s %s\n", lucky(), sign(-4), sign(0));
    return 0;
}

🧪 **Experimente:** chame `lucky(5)` e leia o erro: o `(void)` deixa o compilador pegar isso. Depois apague o `return "zero";` e leia o aviso.

## 4. Argumentos são cópias

Quando você chama uma função, o **valor de cada argumento é copiado** para o parâmetro. A função trabalha com a cópia dela, e a variável de quem chamou não muda. Para devolver um resultado, use `return`. (Para mudar a própria variável de quem chamou, o C usa ponteiros, numa unidade mais à frente.) Os slides mostram a cópia na pilha de chamadas.

#### 🎞️ Argumentos são cópias

![add_one(n) recebe a própria cópia x de n; mudar x deixa o n do main em 5](copy-call-steps.pt-br.svg)

In [ ]:
#include <stdio.h>

void add_one(int x) {
    x++;
    printf("dentro de add_one: x = %d\n", x);
}

int plus_one(int x) {
    return x + 1;
}

int main(void) {
    int n = 5;
    add_one(n);
    printf("depois de add_one: n = %d\n", n);
    n = plus_one(n);
    printf("depois de n = plus_one(n): n = %d\n", n);
    return 0;
}

🧪 **Experimente:** preveja as três linhas antes de rodar. Depois renomeie `x` para `n` em `add_one`: muda alguma coisa? (Não: continua sendo uma cópia separada.)

### ✏️ Exercício: O menor de três

Devolva o menor de três ints. Declare uma auxiliar `min2` e chame-a duas vezes.

In [ ]:
int min2(int a, int b) {
    return a < b ? a : b;
}

int min3(int a, int b, int c) {
    // TODO: devolva min2(min2(a, b), c)
    return a;
}

int main(void) {
    check_int("primeiro", min3(1, 5, 9), 1);
    check_int("meio", min3(5, -2, 9), -2);
    check_int("último", min3(5, 4, 3), 3);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
int min2(int a, int b) {
    return a < b ? a : b;
}

int min3(int a, int b, int c) {
    return min2(min2(a, b), c);
}

int main(void) {
    check_int("primeiro", min3(1, 5, 9), 1);
    check_int("meio", min3(5, -2, 9), -2);
    check_int("último", min3(5, 4, 3), 3);
    return 0;
}
```

</details>

## 5. Variáveis locais, globais e static

Uma variável criada dentro de uma função é **local**: ela é criada a cada chamada e some quando a função termina. Uma variável fora de todas as funções é **global**: qualquer função pode mudá-la, o que dificulta achar bugs, então use pouco. Uma **local `static`** fica no meio: só a função dela a enxerga, mas ela é criada uma vez e **guarda o valor entre as chamadas**.

In [ ]:
#include <stdio.h>

int calls_local(void) {
    int count = 0;           // criada de novo a cada chamada
    count++;
    return count;
}

int calls_static(void) {
    static int count = 0;    // criada uma vez, guarda o valor
    count++;
    return count;
}

int main(void) {
    for (int i = 0; i < 3; i++) {
        printf("local: %d, static: %d\n", calls_local(), calls_static());
    }
    return 0;
}

🧪 **Experimente:** tente imprimir `count` dentro do `main`: nenhuma das duas é visível ali.

### ✏️ Exercício: Um contador que lembra

Cada chamada devolve o próximo id: 1, depois 2, depois 3. Uma local `static` guarda o valor entre as chamadas.

In [ ]:
int next_id(void) {
    // TODO: guarde um contador static e aumente-o
    return 0;
}

int main(void) {
    check_int("primeira chamada", next_id(), 1);
    check_int("segunda chamada", next_id(), 2);
    check_int("terceira chamada", next_id(), 3);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
int next_id(void) {
    static int last = 0;
    last++;
    return last;
}

int main(void) {
    check_int("primeira chamada", next_id(), 1);
    check_int("segunda chamada", next_id(), 2);
    check_int("terceira chamada", next_id(), 3);
    return 0;
}
```

</details>

## 6. Funções que chamam funções, e recursão

Funções chamam outras funções, e a si mesmas, como os métodos do Java. Cada chamada ganha o próprio quadro na pilha de chamadas, com as próprias cópias dos parâmetros e das variáveis locais.

In [ ]:
#include <stdio.h>

int gcd(int a, int b) {
    if (b == 0) {
        return a;               // caso base
    }
    return gcd(b, a % b);
}

int lcm(int a, int b) {
    return a / gcd(a, b) * b;
}

int main(void) {
    printf("mdc(12, 18) = %d\n", gcd(12, 18));
    printf("mmc(4, 6) = %d\n", lcm(4, 6));
    return 0;
}

🧪 **Experimente:** acompanhe `gcd(12, 18)` na mão: gcd(18, 12), gcd(12, 6), gcd(6, 0)… Depois ponha um `printf` no começo de `gcd` para conferir.

## 7. A biblioteca padrão

O C vem com uma biblioteca padrão, dividida em cabeçalhos que você inclui com `#include`: `<stdio.h>` (entrada e saída), `<stdlib.h>` (`abs`, `atoi`, `malloc`…), `<string.h>` (texto), `<ctype.h>` (`isdigit`, `toupper`…), `<stdint.h>` (inteiros de tamanho fixo). O cabeçalho traz os protótipos, então o compilador consegue conferir as suas chamadas.

In [ ]:
#include <ctype.h>
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    printf("%d\n", abs(-12));
    printf("%d\n", atoi("345") + 1);
    printf("%c %d\n", toupper('q'), isdigit('7') != 0);
    return 0;
}

🧪 **Experimente:** tire o `#include <ctype.h>` e leia o erro sobre o `toupper`: sem o cabeçalho, o compilador não conhece a função.

## Em resumo

Uma função em C tem um tipo de retorno, um nome e parâmetros com tipo, como um método `static` do Java sem classe em volta. O C lê o arquivo de cima para baixo, então uma função precisa estar **declarada** antes de ser usada: ou definida acima, ou anunciada por um **protótipo**.

```c
#include <stdio.h>

int next_ticket(void);           // protótipo: definida abaixo

int main(void) {
    for (int i = 0; i < 3; i++) {
        printf("Ticket %d\n", next_ticket());   // 1, 2, 3
    }
    return 0;
}

int next_ticket(void) {
    static int last = 0;          // criada uma vez, mantém o valor entre chamadas
    last++;
    return last;
}
```

Os argumentos são **copiados** para os parâmetros: uma função que muda o parâmetro muda só a cópia dela. Para devolver um resultado, use `return`.

**Cuidado:** escreva `(void)` para "sem parâmetros". Um `()` vazio numa declaração quer dizer "parâmetros não informados" no C17, então o compilador não consegue conferir a chamada.

## Leituras

- [Functions](https://beej.us/guide/bgc/html/split/functions.html) (Beej's Guide to C Programming, chapter 4, 8 min): Passagem por valor, protótipos de funções, e por que int f(void) não é o mesmo que int f() no C mais antigo.
- [Static variables](https://www.freecodecamp.org/news/the-c-beginners-handbook/#heading-static-variables) (freeCodeCamp, The C Beginner's Handbook, 3 min): Opcional: uma variável local static mantém o valor de uma chamada para a outra. A seção "Global variables", logo depois, mostra o outro jeito de compartilhar estado, e por que é mais arriscado.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Rolador de dados** · Fácil
- **Verificação de Luhn** · Fácil
- **Calendário do mês** · Médio
- **Formas de dar troco** · Médio
- **Avaliador de expressões** · Difícil